## This is the implementation of Naive Bayes Algorithm
To classify whether the text is either <b>positive</b> or <b>negative</b> in terms of sentiment.

In [48]:
import numpy as np
import matplotlib as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
print("Setup Complete")

Setup Complete


### Plotting the Data

Importing the data into panda dataframe.

In [ ]:
fields = ["reviews"]
X = pd.read_csv("../Naive-Bayes/dataset/IMDB.csv", usecols=["review"])
y = pd.read_csv("../Naive-Bayes/dataset/IMDB.csv", usecols=["sentiment"])
#Classifying Positive as 0 and negative as 1
y["output_conditional"] = np.where(y["sentiment"] == "positive", 0,1)
# y["output_conditional"] = pd.to_numeric(y["output_conditional"], errors="coerce")
#Splitting the data for test and train
X_train_df, X_test_df, y_train, y_test = train_test_split(X["review"], y["output_conditional"], test_size=0.3, random_state=42)


38094    1
40624    0
49425    1
35734    0
41708    1
Name: output_conditional, dtype: int64
38094    As much as I love trains, I couldn't stomach t...
40624    This was a very good PPV, but like Wrestlemani...
49425    Not finding the right words is everybody's pro...
35734    I'm really suprised this movie didn't get a hi...
41708    I'll start by confessing that I tend to really...
Name: review, dtype: object


### Vectorizing the data

Vectorizing the data for TF-IDF or BoW

In [ ]:
vectorizer = CountVectorizer()
X_train_vectors = vectorizer.fit_transform(X_train_df)
X_test_vectors = vectorizer.transform(X_test_df)
print(f"Vectorized vectors for feature 'reviews'")

Vectorized vectors for feature 'reviews'


In [ ]:
def param_y(expected_output):
    m  = len(expected_output)
    positive_count = sum(expected_output == 1)
    final_sum = positive_count/m
    return final_sum

def params_y_0(expected_output, X):
    expected_output_np = expected_output.to_numpy()
    X_pos = X[expected_output_np == 0]
    X_neg = X[expected_output_np == 1]
    phi_k_y1 = (X_pos.sum(axis=0) + 1) / (X_pos.sum() + len(vectorizer.get_feature_names_out()))
    phi_k_y0 = (X_neg.sum(axis=0) + 1) / (X_neg.sum()+len(vectorizer.get_feature_names_out()))
    return phi_k_y0, phi_k_y1

def predict(phi_y, phi_k_y1, phi_k_y0, X):
    if len(X.shape) == 1:
        X = X.reshape(1, -1)

    predict_y1 = np.log(phi_y) + X.dot(np.log(phi_k_y1.T))
    predict_y0 = np.log(1 - phi_y) + X.dot(np.log(phi_k_y0.T))

    return np.where(predict_y1 > predict_y0, 0, 1)

In [ ]:
phi_y = param_y(y_train)
phi_k_y0, phi_k_y1 = params_y_0(y_train, X_train_vectors)
print(f"Value of phi_y is: {phi_y}")
predictions = predict(phi_y, phi_k_y1, phi_k_y0, X_test_vectors)
print(f"y_test length: {len(y_test)}, predictions length: {len(predictions)}")
accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy * 100:.2f}%")


Value of phi_y is: 0.5025428571428572
y_test length: 15000, predictions length: 15000
✅ Model Accuracy: 84.86%
